# RAG Lab 1 — College Rules Assistant

**Purpose:** Load 4 text documents, chunk them, embed chunks with `all-MiniLM-L6-v2`, index with FAISS, retrieve relevant chunks, generate answers with citations, test answerable/unanswerable questions, and log failures.

> The included college documents are synthetic examples for this lab, not official institutional policy.

## Setup
Run this notebook in Google Colab or locally with Python 3.10+. For local generation with a Hugging Face model, a GPU is recommended. The answer-generation model is `Qwen/Qwen2.5-1.5B-Instruct`; no Anthropic/OpenAI API key is required. The embedding model is separate and is the one specified in the lab handout.

## 0. Install dependencies

If running in Google Colab, run this cell once, then restart the runtime if prompted.

In [1]:
%pip install -q sentence-transformers faiss-cpu transformers accelerate torch

Note: you may need to restart the kernel to use updated packages.


## 1. Imports and configuration

In [2]:
from pathlib import Path
import glob, re, os, time, json
import numpy as np
import faiss
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
from IPython.display import display, Markdown

DATA_DIR = Path("data")
if not DATA_DIR.exists():
    # Helpful fallback if notebook is launched from the parent project directory.
    DATA_DIR = Path("college_rag_lab1_project/data")

print("Data directory:", DATA_DIR.resolve())
print("CUDA available:", torch.cuda.is_available())

/home/swaraj/Desktop/college_rag_lab1_project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Data directory: /home/swaraj/Desktop/college_rag_lab1_project/data
CUDA available: True


## 2. Load and chunk the `.txt` documents

The lab handout suggests chunks of about **250 words with 40-word overlap**. For these four short documents, some documents may produce only one or two chunks; that is expected.

In [3]:
def clean_text(text):
    text = text.replace("\\x0c", " ")
    text = re.sub(r"\\n[ \\t]*\\n+", "\\n\\n", text)
    return text.strip()

def chunk_text(text, size=250, overlap=40):
    words = text.split()
    if not words:
        return []
    if overlap >= size:
        raise ValueError("overlap must be smaller than size")
    step = size - overlap
    return [" ".join(words[i:i + size]) for i in range(0, len(words), step)]

chunks, metadata = [], []
paths = sorted(DATA_DIR.glob("*.txt"))
paths = [p for p in paths if p.name.lower() != "readme.txt"]

if not paths:
    raise FileNotFoundError(f"No .txt documents found in {DATA_DIR.resolve()}")

for path in paths:
    text = clean_text(path.read_text(encoding="utf-8"))
    for chunk_id, piece in enumerate(chunk_text(text, size=250, overlap=40)):
        chunks.append(piece)
        metadata.append({"source": path.name, "local_chunk": chunk_id})

print(f"Loaded {len(paths)} documents and created {len(chunks)} chunks.")
for i, (piece, meta) in enumerate(zip(chunks, metadata)):
    print(f"\\n--- Chunk {i} | {meta['source']} | {len(piece.split())} words ---")
    print(piece[:500] + ("..." if len(piece) > 500 else ""))

Loaded 4 documents and created 6 chunks.
\n--- Chunk 0 | academic_services.txt | 226 words ---
VIT PUNE — SAMPLE ACADEMIC SERVICES AND STUDENT SUPPORT Document type: Student services Status: Fictional sample document created for a RAG lab; not an official VIT Pune policy. Version: 1.0 STUDENT HELP DESK The student help desk operates from 9:30 am to 4:30 pm Monday through Friday, excluding public holidays. Students can ask the help desk for guidance on routine academic administration and how to contact the appropriate department. The document does not provide a telephone number or email ad...
\n--- Chunk 1 | academic_services.txt | 16 words ---
campus security or the relevant emergency service. This sample document does not include emergency phone numbers.
\n--- Chunk 2 | exam_rules.txt | 238 words ---
VIT PUNE — SAMPLE ACADEMIC EXAMINATION RULES Document type: Academic policy Status: Fictional sample document created for a RAG lab; not an official VIT Pune policy. Version: 1.0 ATTENDAN

## 3. Embed chunks and build a FAISS index

The lab specifies `all-MiniLM-L6-v2`, normalized embeddings, and a FAISS inner-product index. With normalized vectors, inner product is equivalent to cosine similarity.

In [4]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
E = embedding_model.encode(chunks, normalize_embeddings=True, convert_to_numpy=True)
E = E.astype("float32")

index = faiss.IndexFlatIP(E.shape[1])
index.add(E)

print("Embedding matrix shape:", E.shape)
print("FAISS vectors indexed:", index.ntotal)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11946.28it/s]


Embedding matrix shape: (6, 384)
FAISS vectors indexed: 6


### Checkpoint: search for “minimum attendance”

In [5]:
def dense_search(question, k=4):
    k = min(k, len(chunks))
    qv = embedding_model.encode([question], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    scores, ids = index.search(qv, k)
    return [(int(i), float(score)) for score, i in zip(scores[0], ids[0]) if i >= 0]

for chunk_id, score in dense_search("minimum attendance", k=3):
    print(f"score={score:.3f} | chunk={chunk_id} | source={metadata[chunk_id]['source']}")
    print(chunks[chunk_id][:250], "\\n")

score=0.318 | chunk=4 | source=hostel_rules.txt
VIT PUNE — SAMPLE HOSTEL RULES Document type: Student accommodation Status: Fictional sample document created for a RAG lab; not an official VIT Pune policy. Version: 1.0 GATE TIMINGS The hostel gate closes at 10:00 pm on weekdays and 11:00 pm on wee \n
score=0.312 | chunk=2 | source=exam_rules.txt
VIT PUNE — SAMPLE ACADEMIC EXAMINATION RULES Document type: Academic policy Status: Fictional sample document created for a RAG lab; not an official VIT Pune policy. Version: 1.0 ATTENDANCE ELIGIBILITY Students must have at least 75% attendance to be \n
score=0.307 | chunk=5 | source=library_rules.txt
VIT PUNE — SAMPLE LIBRARY RULES Document type: Library services Status: Fictional sample document created for a RAG lab; not an official VIT Pune policy. Version: 1.0 OPENING HOURS The library is open from 8:00 am to 8:00 pm Monday through Friday. On \n


## 4. Load an open-source Hugging Face answer-generation model

This notebook uses `Qwen/Qwen2.5-1.5B-Instruct` from Hugging Face. It runs locally and does not require an API key, though first-time model download requires internet.

If model loading is too slow on your computer, run the notebook in Colab with a GPU. You can also replace the model ID with another compatible instruction-tuned causal language model.

In [6]:
GEN_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_ID)
model_kwargs = {"torch_dtype": torch.float16 if torch.cuda.is_available() else torch.float32}
if torch.cuda.is_available():
    model_kwargs["device_map"] = "auto"

generator = AutoModelForCausalLM.from_pretrained(GEN_MODEL_ID, **model_kwargs)
if not torch.cuda.is_available():
    generator = generator.to("cpu")

print("Loaded generation model:", GEN_MODEL_ID)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 341.43it/s]


Loaded generation model: Qwen/Qwen2.5-1.5B-Instruct


## 5. Retrieve context and generate a cited answer

The prompt instructs the model to use only retrieved context, cite claims using `[1]`, `[2]`, etc., and say **“I don't know based on the provided documents.”** when the answer is not supported.

In [8]:
def retrieve(question, k=4):
    hits = dense_search(question, k=k)
    return [
        (chunks[i], metadata[i]["source"], i, score)
        for i, score in hits
    ]


def llm(prompt, max_new_tokens=220):
    messages = [
        {
            "role": "system",
            "content": (
                "You answer questions using only the supplied "
                "document context. Do not use outside knowledge."
            )
        },
        {
            "role": "user",
            "content": prompt
        },
    ]

    rendered = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(rendered, return_tensors="pt")

    if torch.cuda.is_available():
        inputs = {
            key: value.to(generator.device)
            for key, value in inputs.items()
        }

    with torch.no_grad():
        output = generator.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = output[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()


def answer(question, k=4, show_context=True):
    hits = retrieve(question, k=k)

    context = "\n\n".join(
        f"[{n+1}] Source: {source} (chunk {chunk_id})\n{text}"
        for n, (text, source, chunk_id, score) in enumerate(hits)
    )

    prompt = f'''Use ONLY the context below to answer the question.

Cite each factual answer using the bracketed source number,
for example [1].

If the context does not contain the answer, reply exactly:
"I don't know based on the provided documents."

Do not guess or use outside knowledge.

Context:
{context}

Question: {question}

Answer:'''

    response = llm(prompt)

    if show_context:
        print("Retrieved chunks:")

        for n, (text, source, chunk_id, score) in enumerate(hits, 1):
            print(
                f"\n[{n}] {source} | "
                f"chunk {chunk_id} | score={score:.4f}"
            )
            print(text[:350], "\n")

    return response


print(
    answer(
        "What is the minimum attendance required "
        "to appear for the semester-end examination?"
    )
)

Retrieved chunks:

[1] exam_rules.txt | chunk 2 | score=0.4310
VIT PUNE — SAMPLE ACADEMIC EXAMINATION RULES Document type: Academic policy Status: Fictional sample document created for a RAG lab; not an official VIT Pune policy. Version: 1.0 ATTENDANCE ELIGIBILITY Students must have at least 75% attendance to be eligible to appear for the semester-end examination. Students whose attendance is between 65% and b 


[2] academic_services.txt | chunk 0 | score=0.3653
VIT PUNE — SAMPLE ACADEMIC SERVICES AND STUDENT SUPPORT Document type: Student services Status: Fictional sample document created for a RAG lab; not an official VIT Pune policy. Version: 1.0 STUDENT HELP DESK The student help desk operates from 9:30 am to 4:30 pm Monday through Friday, excluding public holidays. Students can ask the help desk for g 


[3] library_rules.txt | chunk 5 | score=0.3570
VIT PUNE — SAMPLE LIBRARY RULES Document type: Library services Status: Fictional sample document created for a RAG lab; not an off

## 6. Ask a few questions manually

Check that the answer is supported by the cited retrieved chunk. A citation is not automatically correct just because the model included one.

In [9]:
manual_questions = [
    "When does the hostel gate close on weekends?",
    "What is the fine for returning a library book late?",
    "How many days do students have to apply for re-evaluation?",
    "What is the MBA tuition fee?"
]

for q in manual_questions:
    print("\\nQUESTION:", q)
    print("ANSWER:", answer(q, k=4, show_context=False))

\nQUESTION: When does the hostel gate close on weekends?
ANSWER: The hostel gate closes at 11:00 pm on weekends.
\nQUESTION: What is the fine for returning a library book late?
ANSWER: ₹2 per book per day
\nQUESTION: How many days do students have to apply for re-evaluation?
ANSWER: Seven calendar days
\nQUESTION: What is the MBA tuition fee?
ANSWER: I don't know based on the provided documents.


## 7. Lab 1 test set: 5 answerable and 3 unanswerable questions



In [10]:
test_questions = [
    {"id": "Q1", "q": "What minimum attendance is required for the semester-end examination?", "answerable": True},
    {"id": "Q2", "q": "When does the hostel gate close on weekends?", "answerable": True},
    {"id": "Q3", "q": "What are the library's weekday opening hours?", "answerable": True},
    {"id": "Q4", "q": "How much is the fine per book per day for a late return?", "answerable": True},
    {"id": "Q5", "q": "How many days after result declaration can a student apply for re-evaluation?", "answerable": True},
    {"id": "Q6", "q": "What is the MBA tuition fee?", "answerable": False},
    {"id": "Q7", "q": "What is today's lunch menu in the hostel mess?", "answerable": False},
    {"id": "Q8", "q": "What is the student help desk's phone number?", "answerable": False},
]

failure_log = []
for item in test_questions:
    print("\\n" + "="*90)
    print(item["id"], "| answerable:", item["answerable"])
    print("QUESTION:", item["q"])
    hits = retrieve(item["q"], k=4)
    print("TOP RETRIEVED SOURCES:", [(h[1], h[2], round(h[3], 3)) for h in hits])
    response = answer(item["q"], k=4, show_context=False)
    print("ANSWER:", response)
    failure_log.append({
        "Question": item["id"] + ": " + item["q"],
        "Answerable?": "Yes" if item["answerable"] else "No",
        "Right chunk retrieved?": "Check manually",
        "Answer correct?": "Check manually",
        "Hallucinated?": "Check manually",
        "Notes": response
    })

import pandas as pd
failure_df = pd.DataFrame(failure_log)
display(failure_df)

\n==========================================================================================
Q1 | answerable: True
QUESTION: What minimum attendance is required for the semester-end examination?
TOP RETRIEVED SOURCES: [('exam_rules.txt', 2, 0.41), ('library_rules.txt', 5, 0.379), ('academic_services.txt', 0, 0.361), ('hostel_rules.txt', 4, 0.327)]
ANSWER: 75%
\n==========================================================================================
Q2 | answerable: True
QUESTION: When does the hostel gate close on weekends?
TOP RETRIEVED SOURCES: [('hostel_rules.txt', 4, 0.53), ('library_rules.txt', 5, 0.273), ('academic_services.txt', 1, 0.205), ('exam_rules.txt', 2, 0.188)]
ANSWER: The hostel gate closes at 11:00 pm on weekends.
\n==========================================================================================
Q3 | answerable: True
QUESTION: What are the library's weekday opening hours?
TOP RETRIEVED SOURCES: [('library_rules.txt', 5, 0.442), ('hostel_rules.txt', 4, 0.339

,Question,Answerable?,Right chunk retrieved?,Answer correct?,Hallucinated?,Notes
0,Q1: What minimum attendance is required for th...,Yes,Check manually,Check manually,Check manually,75%
1,Q2: When does the hostel gate close on weekends?,Yes,Check manually,Check manually,Check manually,The hostel gate closes at 11:00 pm on weekends.
2,Q3: What are the library's weekday opening hours?,Yes,Check manually,Check manually,Check manually,The library is open from 8:00 am to 8:00 pm Mo...
3,Q4: How much is the fine per book per day for ...,Yes,Check manually,Check manually,Check manually,₹2
4,Q5: How many days after result declaration can...,Yes,Check manually,Check manually,Check manually,Seven calendar days
5,Q6: What is the MBA tuition fee?,No,Check manually,Check manually,Check manually,I don't know based on the provided documents.
6,Q7: What is today's lunch menu in the hostel m...,No,Check manually,Check manually,Check manually,I don't know based on the provided documents.
7,Q8: What is the student help desk's phone number?,No,Check manually,Check manually,Check manually,I don't know based on the provided documents.
